<a href="https://colab.research.google.com/github/hamnasz/Urdu-OCR-Project-Code-Saviours-SI-26-Humna-Imran/blob/main/SI26-Week4/SI26_Week4_humna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 4 — Fine-Tune TrOCR on Urdu Dataset

Follows the Week 4 handout's four steps:

1. Load the Pretrained TrOCR Model
2. Set Up Training
3. Evaluate Your Model
4. Save Your Model

**Before running anything:** `Runtime > Change runtime type > GPU`.

One prerequisite cell is added before Step 1 to load the labeled dataset from Weeks 1–3 into `train_dataset`/`test_dataset`, since the handout assumes those already exist.

## Setup

In [1]:
# Pinned versions -- an unpinned `pip install transformers` was pulling in a version
# whose tokenizer backend fails to load microsoft/trocr-base-printed's tokenizer even
# with sentencepiece installed (ValueError: "Couldn't instantiate the backend tokenizer").
# 4.57.6 is a version confirmed to load it correctly.
!pip install transformers==4.57.6 "pillow<12" pandas sentencepiece protobuf --quiet
# torch/torchvision are left alone -- Colab's preinstalled pair is already matched,
# and a bare `pip install torch` here risks pulling a mismatched torchvision.

# If you already hit the tokenizer error before running this pinned install:
# Runtime > Restart session, then run all cells again from the top -- Colab keeps the
# old (unpinned) transformers loaded in memory until the Python process restarts.


In [2]:
import os
from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/hamnasz/Urdu-OCR-Project-Code-Saviours-SI-26-Humna-Imran.git"
DRIVE_ROOT = "/content/drive/MyDrive/Urdu-OCR-Project"
os.makedirs(DRIVE_ROOT, exist_ok=True)
REPO_DIR = os.path.join(DRIVE_ROOT, "urdu-ocr-repo")

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

DATA_DIR = os.path.join(REPO_DIR, "SI26-Week1", "data")
LABELS_PATH = os.path.join(DATA_DIR, "labels.csv")
print("DATA_DIR:", DATA_DIR)
print("labels.csv exists:", os.path.isfile(LABELS_PATH))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATA_DIR: /content/drive/MyDrive/Urdu-OCR-Project/urdu-ocr-repo/SI26-Week1/data
labels.csv exists: True


## Load Your Labeled Dataset (Weeks 1–3)

*(Not in the handout — the handout's Step 2 assumes `train_dataset`/`test_dataset` already exist. This cell builds them from your `labels.csv`.)*

In [3]:
from transformers import TrOCRProcessor
from torch.utils.data import Dataset, random_split
from PIL import Image
import pandas as pd
import torch

MODEL_NAME = "microsoft/trocr-base-printed"
processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
MAX_LENGTH = 128


def resolve_image_path(data_dir, rel_path):
    """labels.csv has a couple of path conventions mixed in from earlier weeks --
    try the direct join first, then fall back to stripping a redundant leading 'data/'."""
    direct = os.path.join(data_dir, rel_path)
    if os.path.isfile(direct):
        return direct
    if rel_path.startswith("data/"):
        stripped = os.path.join(data_dir, rel_path[len("data/"):])
        if os.path.isfile(stripped):
            return stripped
    return direct


class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, processor, data_dir, max_length=MAX_LENGTH):
        data = pd.read_csv(csv_path)
        resolved = data["image"].apply(lambda p: resolve_image_path(data_dir, p))
        has_file = resolved.apply(os.path.isfile)
        n_missing = int((~has_file).sum())
        if n_missing:
            print(f"Skipping {n_missing} rows with no image on disk")
        self.data = data[has_file].reset_index(drop=True)
        self.paths = resolved[has_file].reset_index(drop=True)
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image = Image.open(self.paths.iloc[idx]).convert("RGB")
        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze()
        text = str(self.data.iloc[idx]["text"])
        pad_id = self.processor.tokenizer.pad_token_id
        ids = self.processor.tokenizer(text).input_ids[: self.max_length]
        ids = ids + [pad_id] * (self.max_length - len(ids))
        ids = [t if t != pad_id else -100 for t in ids]
        return {"pixel_values": pixel_values, "labels": torch.tensor(ids)}


dataset = UrduOCRDataset(LABELS_PATH, processor, data_dir=DATA_DIR)
print(f"Dataset loaded: {len(dataset)} samples")

torch.manual_seed(42)
n_test = max(1, int(0.2 * len(dataset)))
n_train = len(dataset) - n_test
train_dataset, test_dataset = random_split(dataset, [n_train, n_test])
print(f"Train: {len(train_dataset)}  Test: {len(test_dataset)}")


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Dataset loaded: 263 samples
Train: 211  Test: 52


## Step 1 — Load the Pretrained TrOCR Model

TrOCR combines a vision encoder (reads the image) with a text decoder (outputs the characters). This loads a version already trained on printed text, then fine-tunes it on your Urdu images — transfer learning.

In [4]:
from transformers import VisionEncoderDecoderModel

# Check if GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("WARNING: No GPU detected.")
    print("Go to Runtime > Change runtime type > GPU")

# Load the pretrained model
# NOTE: the handout has this as 'microsoft/trocr-baseprinted' (missing hyphen) -- that
# isn't a real model on the Hub. The correct id is 'microsoft/trocr-base-printed'
# (MODEL_NAME, set above).
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)
model = model.to(device)

# Configure model for generation
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

print("Model loaded successfully!")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")


Using device: cuda


Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-printed and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded successfully!
Model parameters: 333,921,792


## Step 2 — Set Up Training

A DataLoader wraps your dataset and feeds it to the model in small batches. A batch size of 4 means the model sees 4 images at a time before updating its weights. The optimiser (AdamW) controls how the model adjusts itself after each batch.

In [5]:
from torch.utils.data import DataLoader
# NOTE: the handout says `from transformers import AdamW` -- that raises ImportError on
# current transformers versions (they dropped their own AdamW in favor of PyTorch's).
# torch.optim.AdamW is the standard drop-in replacement.
from torch.optim import AdamW

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4)

# Optimiser
optimizer = AdamW(model.parameters(), lr=5e-5)

print(f"Training batches per epoch: {len(train_loader)}")
print("Ready to train!")


Training batches per epoch: 53
Ready to train!


This cell will take 20–40 minutes to complete. Do not close your browser tab while it runs. Watch the loss number decrease — that means the model is learning.

In [6]:
num_epochs = 3
loss_history = []

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    print(f"\nEpoch {epoch + 1}/{num_epochs}")
    print("-" * 30)

    for batch_idx, batch in enumerate(train_loader):
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loss_history.append(loss.item())
        if batch_idx % 10 == 0:
            print(f"  Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1} complete | Average Loss: {avg_loss:.4f}")

print("\nTraining complete!")
print(f"Training loss went from {loss_history[0]:.4f} to {loss_history[-1]:.4f}")



Epoch 1/3
------------------------------


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
Token indices sequence length is longer than the specified maximum sequence length for this model (2987 > 512). Running this sequence through the model will result in indexing errors


  Batch 0/53 | Loss: 17.1845
  Batch 10/53 | Loss: 5.7470
  Batch 20/53 | Loss: 4.2432
  Batch 30/53 | Loss: 3.8651
  Batch 40/53 | Loss: 3.8882
  Batch 50/53 | Loss: 3.6815
Epoch 1 complete | Average Loss: 5.0345

Epoch 2/3
------------------------------
  Batch 0/53 | Loss: 3.7277
  Batch 10/53 | Loss: 3.6145
  Batch 20/53 | Loss: 3.5487
  Batch 30/53 | Loss: 3.4630
  Batch 40/53 | Loss: 3.5381
  Batch 50/53 | Loss: 3.5319
Epoch 2 complete | Average Loss: 3.6141

Epoch 3/3
------------------------------
  Batch 0/53 | Loss: 3.5521
  Batch 10/53 | Loss: 3.4825
  Batch 20/53 | Loss: 3.6771
  Batch 30/53 | Loss: 3.6219
  Batch 40/53 | Loss: 3.5106
  Batch 50/53 | Loss: 3.5803
Epoch 3 complete | Average Loss: 3.5825

Training complete!
Training loss went from 17.1845 to 3.5375


## Step 3 — Evaluate Your Model

After training, you need to test how well the model performs on images it has never seen before — the test set. The model is set to `eval()` mode so it does not update its weights during this step.

In [7]:
model.eval()
print("=== Model Evaluation on Test Images ===")
print()

correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"]

        generated_ids = model.generate(pixel_values)
        generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)

        # Replace -100 in labels with pad_token_id before decoding
        labels[labels == -100] = processor.tokenizer.pad_token_id
        actual_text = processor.batch_decode(labels, skip_special_tokens=True)

        for pred, actual in zip(generated_text, actual_text):
            total += 1
            if pred.strip() == actual.strip():
                correct += 1
            print(f"Predicted: {pred}")
            print(f"Actual: {actual}")
            print()

accuracy = (correct / total) * 100 if total > 0 else 0
print(f"Accuracy: {accuracy:.1f}% ({correct}/{total} correct)")
print(f"My model accuracy is {accuracy:.1f}%")

=== Model Evaluation on Test Images ===

Predicted: ��������������������
Actual: لاہور پاکستان کا دل ہے

Predicted: ��������������������
Actual: مثبت پاکستانی تاثر دنیا میں اجاگر کرتا رہوں گا، عامر خان

Predicted: ��������������������
Actual: زمبابوے سے سیریز؛ بے اعتبار بیٹنگ نے کوچ کی دھڑکنیں تیز کردیں

Predicted: ��������������������
Actual:  سرورق کہانی سمندر کی رنگ برنگی دنیا سیدہ نازاں جبیں پیارے بچو! یہ کہانی ہے سبز کچھوے کی، جو �

Predicted: ��������������������
Actual: کتاب انسان کی بہترین دوست ہے

Predicted: ��������������������
Actual: پاکستان نے سپرسکسز ٹورنامنٹ میں بھارت کو شکست دے دی

Predicted: ��������������������
Actual:  روشنی روشن اقوال تجربے اور دانائی کا حاصل علم و حکمت کی باتیں حضور اکرم صلی اللہ علیہ وسلم قائدِ اعظم محمد علی

Predicted: ��������������������
Actual: قومی ٹیم کے ناکام بولنگ کوچ مستقبل کے سہانے خواب دکھانے لگے

Predicted: ��������������������
Actual: آئی پی ایل کی افتتاحی تقریب انعقاد سے قبل تنازع کا شکار

Predicted: ��������������������
Actual:  جذ

**What to record:** your final accuracy percentage above, and 3–5 examples from the printed predictions where the model got it wrong. These become your Week 5 discussion points.

## Step 4 — Save Your Model

Colab sessions reset when you close them. Saving your model to Google Drive means you can reload it next week without retraining from scratch.

In [8]:
from google.colab import drive

drive.mount("/content/drive")

save_path = "/content/drive/MyDrive/SI26-urdu-ocr-model"
model.save_pretrained(save_path)
processor.save_pretrained(save_path)

print(f"Model saved to Google Drive: {save_path}")
print("You can load this model again next week without retraining")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model saved to Google Drive: /content/drive/MyDrive/SI26-urdu-ocr-model
You can load this model again next week without retraining


After running this cell, open https://drive.google.com and confirm the folder `SI26-urdu-ocr-model` exists in your Drive before closing Colab.

## Submission Checklist

- **GitHub link to this notebook** — push it to your repo.
- **`My model accuracy is X%`** — printed at the end of the Step 3 cell.
- **`Training loss went from X to X`** — printed at the end of the Step 2 training loop.
- **Screenshot of training output showing loss decreasing** — screenshot the Step 2 training loop's printed batches.
- **3–5 wrong examples** — pick these from the Predicted/Actual pairs printed in Step 3.